# pre_total_final.ipynb

- 목적: 통합 대피소와 지진/해일 전용 대피소 정보를 합쳐 앱이 바로 읽는 최종 대피소 데이터셋을 만든다.
- 입력: `earthquake_shelter_clean_2.csv`, `tsunami_shelter_clean_2.csv`, `project_3_pre_shelter_final.csv`
- 출력: `final_shelter_dataset.csv`
- 메모: 전용 대피소와 통합 대피소를 한 번에 추천 파이프라인으로 넘기기 위한 마지막 병합 단계다.


## 1. 입력 데이터 로딩
최종 병합에 필요한 통합/전용 대피소 CSV를 모두 읽는다.


In [ ]:
# 전용 대피소와 통합 대피소 입력 CSV를 불러온다.
import pandas as pd

eq = pd.read_csv("data/earthquake_shelter_clean_2.csv")
ts = pd.read_csv("data/tsunami_shelter_clean_2.csv")
total = pd.read_csv("data/project_3_pre_shelter_final.csv")


## 2. 전용 대피소 수용인원 통합
지진·해일 전용 대피소의 공통 수용인원 정보를 하나의 표로 만든다.


In [ ]:
# 지진/해일 전용 대피소의 수용인원 정보를 하나로 모은다.
capacity = pd.concat([
    eq[["대피소명", "주소", "수용인원"]],
    ts[["대피소명", "주소", "수용인원"]]
])

capacity = capacity.drop_duplicates(
    subset=["대피소명", "주소"]
)


## 3. 통합 대피소에 수용인원 병합
통합 대피소 행에 전용 대피소 수용인원 정보를 덧붙인다.


In [ ]:
# 통합 대피소 데이터에 전용 대피소 수용인원을 병합한다.
total = total.merge(
    capacity,
    on=["대피소명", "주소"],
    how="left"
)


## 4. 통합/전용 대피소 결합
추천에서 같이 사용할 수 있도록 세 종류 대피소를 하나로 합친다.


In [ ]:
# 통합 대피소와 전용 대피소를 하나의 데이터셋으로 합친다.
all_shelters = pd.concat([
    total,
    eq,
    ts
], ignore_index=True)


In [ ]:
all_shelters = all_shelters.drop_duplicates(
    subset=["대피소명", "주소"]
)


## 5. 최종 컬럼 정리와 저장
중복을 제거하고 앱용 최종 컬럼 구조로 맞춘 뒤 CSV를 저장한다.


In [ ]:
# 앱이 바로 읽을 수 있는 최종 컬럼 순서로 맞춘다.
all_shelters = all_shelters[
    ["대피소명", "주소", "대피소유형", "위도", "경도", "시도", "시군구", "지역", "수용인원"]
]


In [ ]:
# 최종 통합 대피소 데이터셋을 저장한다.
all_shelters.to_csv(
    "data/final_shelter_dataset.csv",
    index=False,
    encoding="utf-8-sig"
)
